In [2]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [19]:
from datasets import load_dataset
import random
import re
import pandas as pd

In [4]:
dataset = load_dataset("bragour/Palestinian_Truth_eng" , split="train")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/322 [00:00<?, ? examples/s]

In [11]:
def remove_watermark(liste):
  for i in range(0,len(liste)):
    liste[i] = liste[i].lower().replace("this ﬁle was downloaded from z library project your gateway to knowledge and culture accessible for everyone z library se singlelogin re go to zlibrary se single login ru oﬃcial telegram channel z access https wikipedia org wiki z library","").replace("translated from arabic to english www onlinedoctranslator com","").replace("www","").strip()
    words = liste[i].split()
    liste[i] = ' '.join(words)
    liste[i] = re.sub(r'[^\w\s]', ' ', liste[i])
    liste[i] = re.sub(r'\s+', ' ', liste[i])
  return liste
rows = remove_watermark(dataset["text"])

In [16]:
def extract_between_words(text):
    start_index = text.find('blog title ')
    if start_index == -1:
      start_index = text.find('title blog ')
    end_index = text.find('author')
    if end_index == -1:
        end_index = text.find('writer')
        if end_index == -1:
            return None
    start_index += len('blog title ')
    return text[start_index:end_index]

def extract_after_a_word(text):
    start_index = text.find('content ')
    if start_index == -1:
      start_index = text.find('date ')
      if start_index == -1:
        return None
      start_index += len('date ')
      content = text[start_index:].split()
      return ' '.join(content[3:])
    start_index += len('content ')
    return text[start_index:]

In [18]:
i=0
new_rows = []
for nb in range(0,len(rows)):
  if 'blog title' in rows[nb] or 'title blog' in rows[nb]:
    inst = extract_between_words(rows[nb])
    output = extract_after_a_word(rows[nb])
    new_rows.append({"INST":inst,"OUTPUT":output})
  else:
    text = rows[nb].split()
    while len(text)>0:
      inst_len = random.randint(30,40)
      output_len = random.randint(90,120)
      num_words = min(inst_len,len(text))
      inst = ' '.join(text[:num_words])
      text = ' '.join(text[num_words:]).split()
      num_words = min(output_len,len(text))
      output = ' '.join(text[:num_words])
      text = ' '.join(text[num_words:]).split()
      if(output == ""):
        new_rows[-1]["OUTPUT"] += ' ' + inst
        break
      new_rows.append({"INST":inst,"OUTPUT":output})
    i+=1
print(i)
print(len(new_rows))



105
29939


In [21]:
for i in range(0,len(new_rows)):
  format = "<s>[INST]"+new_rows[i]["INST"]+"[/INST]"+new_rows[i]["OUTPUT"]+"</s>"
  new_rows[i] = {"text":format}

In [22]:

df = pd.DataFrame(new_rows)

output_file = "dataset_formated.csv"
df.to_csv(output_file, index=False)